In [11]:
import paho.mqtt.client as mqtt
import json
import csv
import time
from datetime import datetime

# MQTT settings
broker = "test.mosquitto.org"
port = 1883
topic = "hackforge/mh26-14/node01/sensor-demo-8274"

# CSV file
file_name = "sensor_data.csv"

# Create CSV file and write column names
with open(file_name, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "timestamp", "node",
        "ax", "ay", "az",
        "gx", "gy", "gz"
    ])

# This function runs when MQTT connects
def on_connect(client, userdata, flags, reason_code, properties):
    print("Connected to MQTT")
    client.subscribe(topic)
    print("Subscribed to sensor topic")

# This function runs whenever a sensor message arrives
def on_message(client, userdata, message):
    data = json.loads(message.payload.decode())

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    row = [
        timestamp,
        data["node"],
        data["ax"], data["ay"], data["az"],
        data["gx"], data["gy"], data["gz"]
    ]

    with open(file_name, "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(row)

    print("Sensor data received:", row)

# Create MQTT client
client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2
)

client.on_connect = on_connect
client.on_message = on_message

# Connect to MQTT broker
client.connect(broker, port)

# Start receiving messages
client.loop_start()

# Collect data for 60 seconds
print("Collecting sensor data for 60 seconds...")

time.sleep(60)

# Stop MQTT connection
client.loop_stop()
client.disconnect()

print("Data collection completed!")
print("Saved in:", file_name)

Connected to MQTT
Subscribed to sensor topic
Sensor data received: ['2026-11-22 13:16:44', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:46', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:48', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:50', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:52', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:54', 'N01', 9830, 5734, 12288, 5633, 8253, 4978]
Sensor data received: ['2026-11-22 13:16:56', 'N01', 9830, 5734, 12288, 3013, 11397, 4978]
Sensor data received: ['2026-11-22 13:16:58', 'N01', 9830, 5734, 12288, 3013, 3799, 4192]
Sensor data received: ['2026-11-22 13:17:00', 'N01', 9830, 5734, 12288, 3013, 3799, 4192]
Sensor data received: ['2026-11-22 13:17:02', 'N01', 20480, 8192, 12288, 3013, 3799, 4192]
Sensor data received: ['2026-11-22 13:17:04', 'N01', 

In [12]:
import pandas as pd

data = pd.read_csv("sensor_data.csv")

print("Total sensor readings:", len(data))

data.head()

Total sensor readings: 15


,timestamp,node,ax,ay,az,gx,gy,gz
0,2026-11-22 13:16:44,N01,9830,5734,12288,5633,8253,4978
1,2026-11-22 13:16:46,N01,9830,5734,12288,5633,8253,4978
2,2026-11-22 13:16:48,N01,9830,5734,12288,5633,8253,4978
3,2026-11-22 13:16:50,N01,9830,5734,12288,5633,8253,4978
4,2026-11-22 13:16:52,N01,9830,5734,12288,5633,8253,4978


In [13]:
# Check the data information
data.info()

# Check for missing values
print("\nMissing values:")
print(data.isnull().sum())

# Display basic statistics
data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   timestamp  15 non-null     str  
 1   node       15 non-null     str  
 2   ax         15 non-null     int64
 3   ay         15 non-null     int64
 4   az         15 non-null     int64
 5   gx         15 non-null     int64
 6   gy         15 non-null     int64
 7   gz         15 non-null     int64
dtypes: int64(6), str(2)
memory usage: 1.1 KB

Missing values:
timestamp    0
node         0
ax           0
ay           0
az           0
gx           0
gy           0
gz           0
dtype: int64


,ax,ay,az,gx,gy,gz
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,10266.866667,6826.400000,12506.400000,3327.400000,3571.933333,5187.600000
std,3094.343384,1441.377555,845.859563,2391.448479,6163.460339,1168.347111
min,6553.000000,5734.000000,12288.000000,-655.000000,-5633.000000,4192.000000
25%,9830.000000,5734.000000,12288.000000,3013.000000,-917.000000,4192.000000
50%,9830.000000,5734.000000,12288.000000,3013.000000,3799.000000,4978.000000
75%,10239.500000,8192.000000,12288.000000,5633.000000,8253.000000,4978.000000
max,20480.000000,9830.000000,15564.000000,5633.000000,11397.000000,7336.000000


In [14]:
# Select sensor values for machine learning
sensor_data = data[["ax", "ay", "az", "gx", "gy", "gz"]]

# Display the selected sensor data
sensor_data.head()

,ax,ay,az,gx,gy,gz
0,9830,5734,12288,5633,8253,4978
1,9830,5734,12288,5633,8253,4978
2,9830,5734,12288,5633,8253,4978
3,9830,5734,12288,5633,8253,4978
4,9830,5734,12288,5633,8253,4978


In [15]:
from sklearn.ensemble import IsolationForest

# Create the machine learning model
model = IsolationForest(
    contamination=0.1,
    random_state=42
)

# Train the model using sensor readings
model.fit(sensor_data)

print("Isolation Forest model trained successfully!")

Isolation Forest model trained successfully!


In [16]:
# Predict sensor condition
predictions = model.predict(sensor_data)

# Add results to the dataset
data["result"] = predictions

# Display the results
data[["ax", "ay", "az", "gx", "gy", "gz", "result"]].head()

,ax,ay,az,gx,gy,gz,result
0,9830,5734,12288,5633,8253,4978,1
1,9830,5734,12288,5633,8253,4978,1
2,9830,5734,12288,5633,8253,4978,1
3,9830,5734,12288,5633,8253,4978,1
4,9830,5734,12288,5633,8253,4978,1


In [17]:
# Check the latest sensor reading
latest_result = predictions[-1]

if latest_result == 1:
    print("NORMAL: Mine sensor condition is stable")
else:
    print("WARNING: Unusual sensor pattern detected!")

In [20]:
normal_count = (predictions == 1).sum()
anomaly_count = (predictions == -1).sum()

print("Normal readings:", normal_count)
print("Anomalous readings:", anomaly_count)

Normal readings: 13
Anomalous readings: 2


In [21]:
data.to_csv("sensor_results.csv", index=False)

print("Sensor results saved successfully.")

Sensor results saved successfully.
